# 09 — Custom toxicity classifier

**Doel:** train je eigen binary toxicity-classifiers op de Kaggle pre-computed
`toxicity_score`. Tweede classifier in je pipeline, naast de sentiment-classifier
uit notebook 08.

**Plan:**
1. Load data en bekijk de `toxicity_score` distributie.
2. Definieer binary target: top 25% van posts = "high toxicity", rest = "low toxicity".
3. Chronologische train/test split.
4. Train drie classifiers: L1-Logistic, RandomForest, optioneel Toxic-BERT.
5. Evalueer en vergelijk.
6. Interpretability: welke woorden voorspellen high toxicity?
7. Save models voor live pipeline (fase 2).

**Verschil met notebook 08:**
- Binary classification ipv 3-class (eenvoudiger, hogere accuracy)
- Class imbalance: ~75% low / 25% high → `class_weight="balanced"` is belangrijk
- Transformer-alternatief: `unitary/toxic-bert` (specifiek toxicity-getrained)


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score, roc_auc_score,
                              precision_recall_curve, average_precision_score)

from src.data.preprocess import clean_text

sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore")

print(f"pandas={pd.__version__}, numpy={np.__version__}")

## 1. Load Kaggle data + bekijk toxicity score distributie


In [ ]:
raw = pd.read_csv("../data/raw/trump_truth_archive.csv", low_memory=False)
raw["created_at"] = pd.to_datetime(raw["created_at"], utc=True, format="ISO8601")
raw = raw.dropna(subset=["text", "toxicity_score"]).reset_index(drop=True)

print(f"Total posts with text + toxicity_score: {len(raw):,}")
print(f"\nToxicity score statistics:")
print(raw["toxicity_score"].describe().round(4))

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(raw["toxicity_score"], bins=80, color="darkorange", edgecolor="white", alpha=0.8)
ax.set_yscale("log")
ax.axvline(raw["toxicity_score"].quantile(0.75), color="red", linestyle="--",
           label=f"75th percentile = {raw['toxicity_score'].quantile(0.75):.3f}")
ax.set_xlabel("toxicity score (0-1)")
ax.set_ylabel("# posts (log scale)")
ax.set_title("Distribution of pre-computed toxicity scores (Kaggle dataset)")
ax.legend()
plt.show()

## 2. Binary target definitie

Toxicity scores zijn heavy-skewed naar lage waarden. Een **median-split** zou
een onnatuurlijke drempel geven (de meeste posts hebben heel lage score).
We gebruiken het **75e percentiel** als drempel: top 25% van posts = "high toxicity",
de rest = "low toxicity".

**Belangrijke threshold-keuze**: de drempel berekenen we ALLEEN op de trainingset,
niet de full dataset. Anders is er sprake van label-leakage van test naar train.


In [ ]:
# Eerst de chronological split, dan threshold op train-set bepalen
TEST_SIZE = 0.20

raw_sorted = raw.sort_values("created_at").reset_index(drop=True)
n_train = int(len(raw_sorted) * (1 - TEST_SIZE))
train = raw_sorted.iloc[:n_train].copy()
test = raw_sorted.iloc[n_train:].copy()

# Threshold = 75e percentiel op TRAIN
THRESHOLD = train["toxicity_score"].quantile(0.75)
print(f"Toxicity threshold (75th percentile of train): {THRESHOLD:.4f}")

# Binary target
train["is_toxic"] = (train["toxicity_score"] > THRESHOLD).astype(int)
test["is_toxic"] = (test["toxicity_score"] > THRESHOLD).astype(int)

# Clean text
train["text_clean"] = train["text"].apply(clean_text)
test["text_clean"] = test["text"].apply(clean_text)
train = train[train["text_clean"].str.split().str.len() >= 5].reset_index(drop=True)
test = test[test["text_clean"].str.split().str.len() >= 5].reset_index(drop=True)

print(f"\nTrain: {len(train):,}  |  high-tox rate: {train['is_toxic'].mean()*100:.1f}%")
print(f"Test:  {len(test):,}  |  high-tox rate: {test['is_toxic'].mean()*100:.1f}%")
print(f"\nCutoff date: {train['created_at'].max().date()}")

## 3. TF-IDF features


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True,
    stop_words="english",
)

X_train = vectorizer.fit_transform(train["text_clean"])
X_test = vectorizer.transform(test["text_clean"])

y_train = train["is_toxic"].values
y_test = test["is_toxic"].values

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Vocab:   {len(vectorizer.vocabulary_):,}")

## 4. Train classifiers


In [ ]:
results = {}

# Dummy baseline
dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)
results["dummy"] = dummy

# L1-Logistic (sneller in binary case dan in 3-class)
print("Training L1-Logistic… (~1-2 min)")
logit = LogisticRegression(
    penalty="l1", C=1.0, solver="liblinear",   # liblinear werkt voor binary
    max_iter=2000, random_state=42, class_weight="balanced",
)
logit.fit(X_train, y_train)
results["logistic_l1"] = logit

# Random Forest
print("Training RF… (~2-5 min)")
rf = RandomForestClassifier(
    n_estimators=200, max_depth=None,
    min_samples_split=5, min_samples_leaf=2,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
rf.fit(X_train, y_train)
results["rf"] = rf

print()
for name, model in results.items():
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="binary")
    auc = roc_auc_score(y_test, proba) if proba is not None else np.nan
    print(f"  {name:12s} acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}")

## 5. Metrics + confusion matrices


In [ ]:
for name, model in results.items():
    pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, pred, target_names=["low_tox", "high_tox"], digits=3))

In [ ]:
# Confusion matrices side-by-side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
labels = ["low_tox", "high_tox"]

for ax, (name, model) in zip(axes, results.items()):
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Oranges",
                xticklabels=labels, yticklabels=labels,
                cbar=False, ax=ax)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="binary")
    ax.set_title(f"{name}\nacc={acc:.3f}  f1={f1:.3f}")
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")

fig.suptitle("Toxicity classifiers — confusion matrices on test set", y=1.02)
fig.tight_layout()
plt.show()

## 6. Optioneel: Toxic-BERT transformer

`unitary/toxic-bert` is een transformer specifiek getraind op toxicity detection
(Jigsaw Toxic Comment Classification). 110M parameters, BERT-base architecture.

**Setup**: vereist `pip install transformers torch` (zit in `[nlp]` extra).

**Run-tijd**: ~10-30 min op CPU voor de test set. M-chip met mps: ~3-8 min.

Skip deze cellen als je tijd wil sparen — de twee TF-IDF baselines zijn voldoende
voor de scriptie. Transformer kun je later als robustness check toevoegen.


In [ ]:
try:
    import torch
    from transformers import pipeline
    DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Transformer available. Device: {DEVICE}")
    TRANSFORMER_AVAILABLE = True
except ImportError:
    print("transformers/torch niet geïnstalleerd. Skip transformer cellen.")
    TRANSFORMER_AVAILABLE = False

In [ ]:
# Run Toxic-BERT op de test set
if TRANSFORMER_AVAILABLE:
    print("Loading unitary/toxic-bert… (eerste keer download ~440MB)")
    toxic_pipe = pipeline(
        "text-classification",
        model="unitary/toxic-bert",
        device=DEVICE,
        truncation=True,
        max_length=512,
    )

    # Toxic-BERT output: label="toxic"/"not_toxic", score=probability
    # We zetten om naar (proba_toxic) en threshold op 0.5
    BATCH_SIZE = 32
    texts = test["text"].tolist()
    proba_toxic = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        outs = toxic_pipe(batch, top_k=None)
        # outs[i] is een lijst met dicts: [{label, score}, ...]
        for o in outs:
            # zoek toxic-label en pak de score
            tox_score = next((d["score"] for d in o if d["label"].lower() == "toxic"), None)
            proba_toxic.append(tox_score if tox_score is not None else 0.0)
        if (i // BATCH_SIZE) % 20 == 0:
            print(f"  {i:,}/{len(texts):,}")

    test["proba_bert_toxic"] = proba_toxic
    pred_bert = (np.array(proba_toxic) > 0.5).astype(int)
    test["pred_bert"] = pred_bert

    acc = accuracy_score(y_test, pred_bert)
    f1 = f1_score(y_test, pred_bert, average="binary")
    auc = roc_auc_score(y_test, proba_toxic)
    print(f"\nToxic-BERT  accuracy={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}")
    print()
    print(classification_report(y_test, pred_bert, target_names=["low_tox", "high_tox"], digits=3))

## 7. Vergelijking alle modellen + ROC curves


In [ ]:
# Verzamel metrics
summary_rows = []
for name, model in results.items():
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    summary_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "f1_binary": f1_score(y_test, pred, average="binary"),
        "auc": roc_auc_score(y_test, proba) if proba is not None else np.nan,
    })

if TRANSFORMER_AVAILABLE and "pred_bert" in test.columns:
    summary_rows.append({
        "model": "toxic_bert",
        "accuracy": accuracy_score(y_test, test["pred_bert"].values),
        "f1_binary": f1_score(y_test, test["pred_bert"].values, average="binary"),
        "auc": roc_auc_score(y_test, test["proba_bert_toxic"].values),
    })

summary_df = pd.DataFrame(summary_rows).set_index("model").round(3)
print(summary_df)

# Bar chart
fig, ax = plt.subplots(figsize=(11, 4.5))
summary_df.plot(kind="bar", ax=ax, color=["#3498db", "#e67e22", "#9b59b6"])
ax.set_title("Toxicity classifier comparison")
ax.set_ylabel("score")
ax.set_xticklabels(summary_df.index, rotation=0)
ax.set_ylim(0, 1)
ax.axhline(0.5, color="red", linewidth=0.7, linestyle="--", label="random AUC")
ax.legend()
plt.tight_layout()
plt.show()

## 8. ROC + Precision-Recall curves

Bij class imbalance (75/25) is een AUC + PR curve informatiever dan losse
accuracy. AUC = 0.85+ is goed. PR-AUC is strenger want het straft false positives
in de minderheidsklasse zwaarder.


In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
for name, model in results.items():
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

if TRANSFORMER_AVAILABLE and "proba_bert_toxic" in test.columns:
    proba = test["proba_bert_toxic"].values
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"toxic_bert (AUC={auc:.3f})", linewidth=2)

axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.7, alpha=0.5)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve")
axes[0].legend()

# Precision-Recall curve
for name, model in results.items():
    proba = model.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, label=f"{name} (AP={ap:.3f})")

if TRANSFORMER_AVAILABLE and "proba_bert_toxic" in test.columns:
    proba = test["proba_bert_toxic"].values
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, label=f"toxic_bert (AP={ap:.3f})", linewidth=2)

axes[1].axhline(y_test.mean(), color="red", linestyle="--", linewidth=0.7,
                label=f"baseline (pos rate)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve")
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Top features voor "high toxicity"

L1-Logistic geeft sparse coefficients. Hoogste positieve coefficients =
woorden die het sterkst voor `high_tox` voorspellen.


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = logit.coef_[0]

# Top positive (= high toxicity)
top_pos_idx = np.argsort(coefs)[::-1][:25]
# Top negative (= low toxicity)
top_neg_idx = np.argsort(coefs)[:25]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

axes[0].barh(feature_names[top_pos_idx][::-1], coefs[top_pos_idx][::-1],
             color="#e67e22", edgecolor="white")
axes[0].set_title("Top-25 features → HIGH toxicity")
axes[0].set_xlabel("log-odds coefficient")

axes[1].barh(feature_names[top_neg_idx][::-1], coefs[top_neg_idx][::-1],
             color="#27ae60", edgecolor="white")
axes[1].set_title("Top-25 features → LOW toxicity")
axes[1].set_xlabel("log-odds coefficient (negative)")

fig.tight_layout()
plt.show()

## 10. Voorbeelden van zeer toxic / niet-toxic posts (test set)

Visuele sanity check: ziet de classifier zinvolle posts?


In [ ]:
# Posts met hoogste predicted toxicity probability
test["proba_high_tox"] = logit.predict_proba(X_test)[:, 1]

print("=== Top-5 MOST TOXIC volgens classifier (test set) ===\n")
top_tox = test.nlargest(5, "proba_high_tox")[["created_at", "proba_high_tox", "is_toxic", "text"]]
for _, row in top_tox.iterrows():
    print(f"[{row['created_at'].date()}]  pred={row['proba_high_tox']:.3f}  actual={'TOXIC' if row['is_toxic'] else 'low'}")
    print(f"  {row['text'][:200]}")
    print()

print("\n=== Top-5 MOST NOT-TOXIC volgens classifier ===\n")
not_tox = test.nsmallest(5, "proba_high_tox")[["created_at", "proba_high_tox", "is_toxic", "text"]]
for _, row in not_tox.iterrows():
    print(f"[{row['created_at'].date()}]  pred={row['proba_high_tox']:.3f}  actual={'TOXIC' if row['is_toxic'] else 'low'}")
    print(f"  {row['text'][:200]}")
    print()

## 11. Save trained models


In [ ]:
models_dir = Path("../models/toxicity")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(vectorizer, models_dir / "vectorizer.joblib")
joblib.dump(results["logistic_l1"], models_dir / "logistic_l1.joblib")
joblib.dump(results["rf"], models_dir / "rf.joblib")

metadata = {
    "task": "toxicity_binary_classification",
    "threshold": float(THRESHOLD),
    "threshold_method": "75th percentile of training set toxicity_score",
    "positive_class": "high_tox",
    "train_size": len(train),
    "test_size": len(test),
    "train_pos_rate": float(y_train.mean()),
    "test_pos_rate": float(y_test.mean()),
    "cutoff_date": train["created_at"].max().isoformat(),
    "metrics": summary_df.to_dict(),
    "features": {
        "vectorizer": "TfidfVectorizer(max_features=8000, ngram=(1,2), min_df=5, stop_words=english)",
        "n_features": X_train.shape[1],
    },
}
with open(models_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Saved to {models_dir.resolve()}:")
for p in sorted(models_dir.iterdir()):
    print(f"  {p.name:25s}  {p.stat().st_size:>10,} bytes")

## 12. Bevindingen & next steps

**Wat we hebben** (vul aan met je echte cijfers):
- L1-Logistic: acc __, F1 __, AUC __
- RF: acc __, F1 __, AUC __
- Toxic-BERT: acc __, F1 __, AUC __ (als gerund)
- Threshold: top 25% van train = `high_tox`
- Pos rate in test: __% (vergelijkbaar met train, dus geen drift)

**Voor je scriptie:**

Toxicity classification is een **kleinere** taak dan sentiment — kleinere
positive class, vaak duidelijker lexicale signalen (scheldwoorden, beschuldigingen).
Verwacht hier hogere accuracy en AUC dan voor 3-class sentiment.

Combineer met sentiment-classifier voor het complete verhaal:

> *"We trained two custom classifiers on Kaggle pre-computed labels: sentiment
> (3-class, accuracy 0.83) and toxicity (binary, accuracy 0.XX). Together they
> form the analytical core of our pipeline, which we apply to new posts in
> the live-scraping module (Section X)."*

**Volgende stappen (fase 2):**
- `src/data/live_scraper.py` — truthbrush wrapper met SQLite store voor
  incrementele updates.
- `src/data/score_new_posts.py` — laadt onze sentiment + toxicity classifiers
  en scoort nieuwe posts automatisch.
- Notebook 10 (optioneel) — demo van end-to-end live pipeline.

**Voor de presentatie:**
- Bar chart vergelijking + ROC curves zijn visueel sterk voor slides.
- De top-25 high-toxicity features als interpreteerbaar artefact.
- Voorbeelden van geclassificeerde posts (sectie 10) maken het tastbaar.
